# Load one more field of a partial entity

The order identifier was loaded first and the total later. A copy with the total holds it as loaded; the currency, still not loaded, still cannot be read. A partial entity given every field is the same as the full one.

Use the repository environment for this branch. Run cells in order in a fresh kernel; no published-package install is needed.

## Import the types

In [ ]:
!pip install -q aoa-action-machine

In [2]:
from pydantic import Field

from aoa.action_machine.domain import BaseCoreEntity, FieldNotLoadedError
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entities import entities
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources.base_storage import BaseStorage

## Declare the model

In [3]:
class ShopDomain(BaseDomain):
    """Group the shop model declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Customer order", domain=ShopDomain)
class OrderCoreEntity(BaseCoreEntity):
    """Describe an order with a total and a currency."""

    id: str = Field(description="Order identifier")
    total: float = Field(ge=0, description="Order total")
    currency: str = Field(default="RUB", description="Currency code")


@meta(description="Stores the shop entities")
@entities(
    OrderCoreEntity,
)
class ShopStorage(BaseStorage):
    """Hold the shop entities."""

## Add the total with model_copy

In [4]:
order = OrderCoreEntity.partial(id="ord-5")
updated = order.model_copy(update={"total": 42.0})
print("total loaded:", updated.is_field_loaded("total"), updated.total)
print("currency loaded:", updated.is_field_loaded("currency"))
try:
    print(updated.currency)
except FieldNotLoadedError as exc:
    print("reading currency:", type(exc).__name__)
else:
    raise AssertionError("An unloaded currency must not be readable")

total loaded: True 42.0
currency loaded: False
reading currency: FieldNotLoadedError


## Compare with the full order

In [5]:
every = OrderCoreEntity.partial(id="ord-5", total=42.0, currency="EUR")
full = OrderCoreEntity(id="ord-5", total=42.0, currency="EUR")
print("partial with every field == full:", every == full)
print("partial with the id only == full:", order == full)

partial with every field == full: True
partial with the id only == full: False


`model_copy(update=...)` puts the updated field into the instance, so it is loaded. Equality compares the fields an instance holds, which is why a partial entity with every field equals the full one.